## Installation

In [2]:
!pip install datasets transformers sentencepiece -q

In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

## 1. Load and Inspect TinyStories in Colab

In [4]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories", split="train")

print(ds)
print(ds[0])

Dataset({
    features: ['text'],
    num_rows: 2119719
})
{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [5]:
import numpy as np

lengths = [len(x["text"].split()) for x in ds.select(range(1000))]
print("Avg words:", np.mean(lengths))
print("Max words:", np.max(lengths))

Avg words: 183.801
Max words: 837


## 2. Training our own BPE

Training our own BPE ensures:

- Tokenizer matches dataset distribution

- Model capacity is allocated efficiently

- Replication experiment remains controlled

- Small models are not handicapped by oversized vocabularies

In small-scale experiments, tokenizer choice is not minor — it materially affects model behavior.

2.1 Write texts into a plain text corpus file

In [5]:
out_path = "tiny_corpus.txt"
with open(out_path, "w", encoding="utf-8") as f:
    for ex in ds:
        txt = ex.get("text", "")
        if txt:
            f.write(txt.replace("\r\n", "\n").replace("\r", "\n").strip() + "\n")

print("Wrote:", out_path)

Wrote: tiny_corpus.txt


2.2 Train the BPE tokenizer

In [6]:
import os
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer()
v_size = 1500 # Adjustable

tokenizer.train(
    files=["tiny_corpus.txt"],
    vocab_size=v_size,
    min_frequency=2,
    special_tokens=["<s>", "<pad>", "</s>", "<unk>"],
)

save_directory = "./"
os.makedirs(save_directory, exist_ok=True)

# Update file name accordingly to save in the specified drive folder
tokenizer.save_model(save_directory)
print(f"Saved to {save_directory}")




Saved to ./


#### 2.3 Load saved BPE tokenizer

In [7]:
import os

save_directory = "./"

print("Files in directory:")
print(os.listdir(save_directory))

Files in directory:
['.config', '.profile', '.ipynb_checkpoints', 'tokenizer.json', '.nv', '.zshrc', 'runs', 'vocab.json', '.verb-setup.log', '.bash_logout', '.ssh', '.bash_history', 'tiny_corpus.txt', 'experiment_log.json', 'TinyStories_SLM.ipynb', 'tinystories_lm_val_512', 'models', '.venv', '.cache', '.ipython', 'tinystories_lm_train_512', 'merges.txt', '.sudo_as_admin_successful', '.jupyter', '.local', '.bashrc']


In [8]:
from tokenizers import ByteLevelBPETokenizer
import os

save_directory = "./"
vocab_path  = os.path.join(save_directory, "vocab.json")
merges_path = os.path.join(save_directory, "merges.txt")

bpe = ByteLevelBPETokenizer(vocab_path, merges_path)

# Save a unified tokenizer file
bpe.save(os.path.join(save_directory, "tokenizer.json"))
print("Saved tokenizer.json")

Saved tokenizer.json


In [6]:
from transformers import PreTrainedTokenizerFast
import os

save_directory = "./"

tok = PreTrainedTokenizerFast(
    tokenizer_file=os.path.join(save_directory, "tokenizer.json"),
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
)

print("Vocab size:", tok.vocab_size)
print(tok("Once upon a time.")["input_ids"])

Vocab size: 8000
[434, 450, 262, 399, 17]


#### 2.4 Decide context length

In [10]:
from datasets import load_dataset
import numpy as np

ds = load_dataset("roneneldan/TinyStories", split="train")

lens = [len(tok(t)["input_ids"]) for t in ds["text"][:2000]]
print("Avg tokens:", float(np.mean(lens)))
print("P95 tokens:", float(np.percentile(lens, 95)))
print("Max tokens:", int(np.max(lens)))

Avg tokens: 250.6945
P95 tokens: 462.0999999999999
Max tokens: 1068


From what observed:
- On average, each story is about 206 tokens
- 95% out of 2000 stories are shorter than 392 tokens
-> We can choose context length (block_size) of 512 for the models since it is enough to fit the whole story in

## 3. Training the model

#### 3.1 Build packed dataset

In [11]:
raw = load_dataset("roneneldan/TinyStories")
train_ds = raw["train"]
val_ds = raw["validation"] if "validation" in raw else train_ds.select(range(2000))

block_size = 512

In [12]:
def tokenize_fn(batch):
    out = tok(batch["text"], add_special_tokens=False, return_attention_mask=False)
    # out only has "input_ids"
    return out

tok_train = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
tok_val   = val_ds.map(tokenize_fn, batched=True, remove_columns=val_ds.column_names)

def group_texts(examples):
    concatenated = []
    for ids in examples["input_ids"]:
        concatenated.extend(ids)

    total_len = (len(concatenated) // block_size) * block_size
    concatenated = concatenated[:total_len]

    input_ids = [concatenated[i:i+block_size] for i in range(0, total_len, block_size)]
    return {"input_ids": input_ids, "labels": input_ids.copy()}

lm_train = tok_train.map(group_texts, batched=True, remove_columns=tok_train.column_names)
lm_val   = tok_val.map(group_texts, batched=True, remove_columns=tok_val.column_names)

print("Packed train blocks:", len(lm_train))
print("Packed val blocks:", len(lm_val))
print("One block length:", len(lm_train[0]["input_ids"]))

Packed train blocks: 1097510
Packed val blocks: 11040
One block length: 512


In [13]:
lm_train.save_to_disk("./tinystories_lm_train_512")
lm_val.save_to_disk("./tinystories_lm_val_512")

Saving the dataset (1/1 shards): 100%|██████████| 11040/11040 [00:00<00:00, 114038.25 examples/s]


#### 3.2 Load Packed Dataset

In [8]:
from datasets import load_from_disk

lm_train = load_from_disk("./tinystories_lm_train_512")
lm_val   = load_from_disk("./tinystories_lm_val_512")

print(len(lm_train), len(lm_val))

906933 9113


In [15]:
print("tok.vocab_size:", tok.vocab_size)
print("len(tok):", len(tok))
print("bos/eos/unk/pad:", tok.bos_token_id, tok.eos_token_id, tok.unk_token_id, tok.pad_token_id)

tok.vocab_size: 1500
len(tok): 1500
bos/eos/unk/pad: 0 2 3 1


#### 3.3 Define a Small Sanity GPT Model (~7M)

In [17]:
import torch
import torch.nn.functional as F
from transformers import (
    GPT2Config, GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    TrainingArguments, Trainer
)
import json, os
from lion_pytorch import Lion

# ============ EXPERIMENT CONFIG ============
LEARNING_RATE   = 5e-4          # try: 1e-3, 5e-4, 3e-4, 1e-4
OPTIMIZER_NAME  = "adamw"       # try: "adamw", "adam", "sgd"
LABEL_SMOOTHING = 0           # try: 0.0, 0.1, 0.2
RUN_NAME        = "baseline_model_1500"
# ===========================================

data_collator = DataCollatorForLanguageModeling(tok, mlm=False)

config = GPT2Config(
    vocab_size=1500,
    n_positions=512,
    n_ctx=512,
    n_embd=256,
    n_layer=6,
    n_head=8,
    bos_token_id=tok.bos_token_id,
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.pad_token_id,
)

model = GPT2LMHeadModel(config)
print("Total parameters:", sum(p.numel() for p in model.parameters())/1e6, "M")


# ============ CUSTOM TRAINER ============
class CustomTrainer(Trainer):

    def create_optimizer(self):
        if OPTIMIZER_NAME == "adamw":
            self.optimizer = torch.optim.AdamW(
                self.model.parameters(),
                lr=self.args.learning_rate,
                weight_decay=self.args.weight_decay,
            )
        elif OPTIMIZER_NAME == "adam":
            self.optimizer = torch.optim.Adam(
                self.model.parameters(),
                lr=self.args.learning_rate,
            )
        elif OPTIMIZER_NAME == "sgd":
            self.optimizer = torch.optim.SGD(
                self.model.parameters(),
                lr=self.args.learning_rate,
                momentum=0.9,
                weight_decay=self.args.weight_decay,
            )
        elif OPTIMIZER_NAME == "lion":
            self.optimizer = Lion(
                self.model.parameters(),
                lr=self.args.learning_rate / 3,  # Lion needs ~3x smaller LR than Adam
                weight_decay=self.args.weight_decay,
            )
        return self.optimizer

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        shift_logits = outputs.logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100,
            label_smoothing=LABEL_SMOOTHING,
        )
        return (loss, outputs) if return_outputs else loss
# ========================================


training_args = TrainingArguments(
    output_dir=f"./runs/{RUN_NAME}",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,
    num_train_epochs=1,
    learning_rate=LEARNING_RATE,
    warmup_steps=500,              
    weight_decay=0.1,
    fp16=True,
    report_to="none",
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
    data_collator=data_collator,
)

trainer.train()

# ============ LOG RESULTS ============
eval_logs = [x for x in trainer.state.log_history if "eval_loss" in x]

if eval_logs:
    val_loss = eval_logs[-1]["eval_loss"]
    perplexity = torch.exp(torch.tensor(val_loss)).item()

    log_entry = {
        "run_name"        : RUN_NAME,
        "learning_rate"   : LEARNING_RATE,
        "optimizer"       : OPTIMIZER_NAME,
        "label_smoothing" : LABEL_SMOOTHING,
        "final_val_loss"  : val_loss,
        "perplexity"      : perplexity,
    }

    log_path = "./experiment_log.json"
    all_logs = json.load(open(log_path)) if os.path.exists(log_path) else []
    all_logs.append(log_entry)
    json.dump(all_logs, open(log_path, "w"), indent=2)

    print(f"\nRun:        {RUN_NAME}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Perplexity: {perplexity:.2f}")
else:
    print("No eval logs found — check eval_steps vs total training steps")
# =====================================

Total parameters: 5.254144 M


Step,Training Loss,Validation Loss
500,7.110095,3.364768
1000,5.357430,2.472102
1500,4.597120,2.136003
2000,4.276356,1.993554
2500,4.095283,1.910250
3000,3.974152,1.856530
3500,3.901405,1.811520
4000,3.821845,1.780846
4500,3.761152,1.753722
5000,3.702007,1.731434


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 35.99it/s]



Run:        baseline_model_1500
Val Loss:   1.5373
Perplexity: 4.65


In [18]:
import torch
import torch.nn.functional as F
from transformers import (
    GPT2Config, GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    TrainingArguments, Trainer
)
import json, os
from lion_pytorch import Lion

# ============ EXPERIMENT CONFIG ============
LEARNING_RATE   = 1e-3          # try: 1e-3, 5e-4, 3e-4, 1e-4
OPTIMIZER_NAME  = "adam"       # try: "adamw", "adam", "sgd"
LABEL_SMOOTHING = 0           # try: 0.0, 0.1, 0.2
RUN_NAME        = "best_model"
# ===========================================

data_collator = DataCollatorForLanguageModeling(tok, mlm=False)

config = GPT2Config(
    vocab_size=1500,
    n_positions=512,
    n_ctx=512,
    n_embd=768,
    n_layer=12,
    n_head=12,
    bos_token_id=tok.bos_token_id,
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.pad_token_id,
)

model = GPT2LMHeadModel(config)
print("Total parameters:", sum(p.numel() for p in model.parameters())/1e6, "M")


# ============ CUSTOM TRAINER ============
class CustomTrainer(Trainer):

    def create_optimizer(self):
        if OPTIMIZER_NAME == "adamw":
            self.optimizer = torch.optim.AdamW(
                self.model.parameters(),
                lr=self.args.learning_rate,
                weight_decay=self.args.weight_decay,
            )
        elif OPTIMIZER_NAME == "adam":
            self.optimizer = torch.optim.Adam(
                self.model.parameters(),
                lr=self.args.learning_rate,
            )
        elif OPTIMIZER_NAME == "sgd":
            self.optimizer = torch.optim.SGD(
                self.model.parameters(),
                lr=self.args.learning_rate,
                momentum=0.9,
                weight_decay=self.args.weight_decay,
            )
        elif OPTIMIZER_NAME == "lion":
            self.optimizer = Lion(
                self.model.parameters(),
                lr=self.args.learning_rate / 3,  # Lion needs ~3x smaller LR than Adam
                weight_decay=self.args.weight_decay,
            )
        return self.optimizer

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        shift_logits = outputs.logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100,
            label_smoothing=LABEL_SMOOTHING,
        )
        return (loss, outputs) if return_outputs else loss
# ========================================


training_args = TrainingArguments(
    output_dir=f"./runs/{RUN_NAME}",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,
    num_train_epochs=1,
    learning_rate=LEARNING_RATE,
    warmup_steps=500,              
    weight_decay=0.1,
    fp16=True,
    report_to="none",
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
    data_collator=data_collator,
)

trainer.train()

# Save final model + tokenizer
save_path = f"./runs/{RUN_NAME}/final_model"

trainer.save_model(save_path)   # saves model + trainer config
tok.save_pretrained(save_path)  # saves tokenizer files

print(f"Model saved to: {save_path}")

# ============ LOG RESULTS ============
eval_logs = [x for x in trainer.state.log_history if "eval_loss" in x]

if eval_logs:
    val_loss = eval_logs[-1]["eval_loss"]
    perplexity = torch.exp(torch.tensor(val_loss)).item()

    log_entry = {
        "run_name"        : RUN_NAME,
        "learning_rate"   : LEARNING_RATE,
        "optimizer"       : OPTIMIZER_NAME,
        "label_smoothing" : LABEL_SMOOTHING,
        "final_val_loss"  : val_loss,
        "perplexity"      : perplexity,
    }

    log_path = "./experiment_log.json"
    all_logs = json.load(open(log_path)) if os.path.exists(log_path) else []
    all_logs.append(log_entry)
    json.dump(all_logs, open(log_path, "w"), indent=2)

    print(f"\nRun:        {RUN_NAME}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Perplexity: {perplexity:.2f}")
else:
    print("No eval logs found — check eval_steps vs total training steps")
# =====================================

Total parameters: 86.601216 M


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [20]:
import torch
import torch.nn.functional as F
from transformers import (
    GPTNeoConfig, GPTNeoForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments, Trainer
)
import json, os

# ============ EXPERIMENT CONFIG ============
LEARNING_RATE   = 5e-4
OPTIMIZER_NAME  = "adamw"
LABEL_SMOOTHING = 0.0
RUN_NAME        = "gptneo_33M_replicated"
# ===========================================

data_collator = DataCollatorForLanguageModeling(tok, mlm=False)

# ============ REPLICATED GPT-NEO ARCHITECTURE ============
# Exact config from TinyStories-33M
# Only change: vocab_size 50,257 → 8,000 (your tokenizer)
config = GPTNeoConfig(
    vocab_size              = tok.vocab_size,   # 8,000 ← only change
    hidden_size             = 768,              # same as theirs
    num_layers              = 4,                # same as theirs
    attention_types         = [[["global", "local"], 2]],  # same as theirs
    num_heads               = 16,               # same as theirs
    intermediate_size       = 3072,
    window_size             = 256,              # same as theirs
    max_position_embeddings = 2048,              # changed from 2048 → your context
    embed_dropout           = 0,               # same as theirs
    attention_dropout       = 0,               # same as theirs
    resid_dropout           = 0,               # same as theirs
    layer_norm_epsilon      = 1e-5,            # same as theirs
    initializer_range       = 0.02,            # same as theirs
    bos_token_id            = tok.bos_token_id,
    eos_token_id            = tok.eos_token_id,
)

model = GPTNeoForCausalLM(config)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Total parameters: {total_params:.1f}M")
# =========================================================


# ============ CUSTOM TRAINER ============
class CustomTrainer(Trainer):

    def create_optimizer(self):
        if OPTIMIZER_NAME == "adamw":
            self.optimizer = torch.optim.AdamW(
                self.model.parameters(),
                lr=self.args.learning_rate,
                weight_decay=self.args.weight_decay,
            )
        elif OPTIMIZER_NAME == "adam":
            self.optimizer = torch.optim.Adam(
                self.model.parameters(),
                lr=self.args.learning_rate,
            )
        return self.optimizer

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        shift_logits = outputs.logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100,
            label_smoothing=LABEL_SMOOTHING,
        )
        return (loss, outputs) if return_outputs else loss
# ========================================


# ============ TRAINING ARGUMENTS ============
training_args = TrainingArguments(
    output_dir=f"./runs/{RUN_NAME}",

    # Identical to your other runs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,
    num_train_epochs=1,
    fp16=True,
    report_to="none",

    # Standard defaults
    learning_rate=LEARNING_RATE,
    warmup_steps=500,
    weight_decay=0.01,
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
    data_collator=data_collator,
)

trainer.train()
# ============================================


# Save model
save_path = f"./runs/{RUN_NAME}/final_model"
trainer.save_model(save_path)
tok.save_pretrained(save_path)
print(f"Model saved to: {save_path}")


# ============ LOG RESULTS ============
eval_logs = [x for x in trainer.state.log_history if "eval_loss" in x]

if eval_logs:
    val_loss   = eval_logs[-1]["eval_loss"]
    perplexity = torch.exp(torch.tensor(val_loss)).item()

    log_entry = {
        "run_name"        : RUN_NAME,
        "learning_rate"   : LEARNING_RATE,
        "optimizer"       : OPTIMIZER_NAME,
        "label_smoothing" : LABEL_SMOOTHING,
        "architecture"    : {
            "model_type"  : "GPT-Neo (replicated TinyStories-33M)",
            "hidden_size" : 768,
            "num_layers"  : 4,
            "num_heads"   : 16,
            "window_size" : 256,
            "vocab"       : tok.vocab_size,
            "params"      : f"{total_params:.1f}M"
        },
        "note"           : "Replicated TinyStories-33M architecture, custom vocab, 1 epoch",
        "final_val_loss" : val_loss,
        "perplexity"     : perplexity,
    }

    log_path  = "./experiment_log.json"
    all_logs  = json.load(open(log_path)) if os.path.exists(log_path) else []
    all_logs.append(log_entry)
    json.dump(all_logs, open(log_path, "w"), indent=2)

    print(f"\n{'='*50}")
    print(f"RESULTS")
    print(f"{'='*50}")
    print(f"Model:      GPT-Neo TinyStories-33M replicated")
    print(f"Params:     {total_params:.1f}M")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Perplexity: {perplexity:.2f}")
    print(f"{'='*50}")

    print(f"\nCOMPARISON:")
    print(f"  GPT-Neo replicated: perplexity {perplexity:.2f}")
    print(f"  Your 91.7M ablation: perplexity 3.81")
    if perplexity > 3.81:
        print(f"  → Your 91.7M wins ✅")
    else:
        print(f"  → GPT-Neo replicated wins")
else:
    print("No eval logs found")

Total parameters: 31.1M


Step,Training Loss,Validation Loss
500,5.761035,2.738801
1000,4.019698,1.968416
1500,3.568680,1.777834
2000,3.345538,1.668972
2500,3.205081,1.602432
3000,3.113345,1.558856
3500,3.056219,1.521349
4000,2.987985,1.491413
4500,2.930857,1.467189
5000,2.880302,1.445736


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.05it/s]

Model saved to: ./runs/gptneo_33M_replicated/final_model

RESULTS
Model:      GPT-Neo TinyStories-33M replicated
Params:     31.1M
Val Loss:   1.2279
Perplexity: 3.41

COMPARISON:
  GPT-Neo replicated: perplexity 3.41
  Your 91.7M ablation: perplexity 3.81
  → GPT-Neo replicated wins


#### 3.5 Test generation

In [20]:
prompt = "What is your name?"
inputs = tok(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        top_p=0.95,
        temperature=0.9,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
    )

print(tok.decode(output[0], skip_special_tokens=True))

What is your name? This is my new dog. His name is Spot." The new dog barks and licks his face. Lily and the new dog's nose. They are very happy.

Sara and Ben play with the new dog. They throw the new dog and throw the new dog down. They make the new dog go over and over. They run and hug the new dog. They play with the new dog and the new dog. They are very happy.Lily and Ben were playing in the park. They saw a big tree with many leaves. They wanted to touch the leaves and the leaves. Mom said


#### 3.6 Check n-grams

In [22]:
from datasets import load_dataset
import random

def get_ngrams(tokens, n):
    return set(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def ngram_containment(generated_text, tok, train_texts, n=8, max_train_docs=20000, seed=42):
    random.seed(seed)
    sample_texts = train_texts if len(train_texts) <= max_train_docs else random.sample(train_texts, max_train_docs)

    # tokenize generation
    gen_ids = tok(generated_text, add_special_tokens=False)["input_ids"]
    if len(gen_ids) < n:
        return {"n": n, "containment": 0.0, "gen_ngrams": 0, "train_docs_used": len(sample_texts)}
    gen_ngrams = get_ngrams(gen_ids, n)

    # build a set of all train ngrams (sampled docs)
    train_ngrams = set()
    for t in sample_texts:
        ids = tok(t, add_special_tokens=False)["input_ids"]
        if len(ids) >= n:
            train_ngrams |= get_ngrams(ids, n)

    overlap = len(gen_ngrams & train_ngrams)
    containment = overlap / max(1, len(gen_ngrams))

    return {
        "n": n,
        "containment": containment,
        "gen_ngrams": len(gen_ngrams),
        "train_ngrams": len(train_ngrams),
        "train_docs_used": len(sample_texts),
        "overlap_ngrams": overlap
    }

# ---- usage ----
raw = load_dataset("roneneldan/TinyStories", split="train")
train_texts = raw["text"]

# generated = """PASTE YOUR GENERATED OUTPUT HERE"""
# generated = tok.decode(output[0], skip_special_tokens=True)
full = tok.decode(output[0], skip_special_tokens=True)
generated = full[len(prompt):].strip()

for n in [4, 8]:
    stats = ngram_containment(generated, tok, train_texts, n=n, max_train_docs=20000)
    print(stats)

{'n': 4, 'containment': 0.5428571428571428, 'gen_ngrams': 105, 'train_ngrams': 2087406, 'train_docs_used': 20000, 'overlap_ngrams': 57}
{'n': 8, 'containment': 0.1415929203539823, 'gen_ngrams': 113, 'train_ngrams': 3908510, 'train_docs_used': 20000, 'overlap_ngrams': 16}


#### 3.7 Save model

In [23]:
save_path = "/content/drive/MyDrive/llm_folder/models/tinystories_sanity"

trainer.save_model(save_path)      # saves model + config
tok.save_pretrained(save_path)     # saves tokenizer files

print("Saved to:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/drive/MyDrive/llm_folder/models/tinystories_sanity


#### 3.8 Load model

In [29]:
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast

load_path = "/content/drive/MyDrive/llm_folder/models/tinystories_sanity"

model2 = GPT2LMHeadModel.from_pretrained(load_path)
tok2 = PreTrainedTokenizerFast.from_pretrained(load_path)

prompt = "What date is it today"
inputs = tok2(prompt, return_tensors="pt")
out = model2.generate(**inputs, max_new_tokens=80, do_sample=True, top_p=0.95, temperature=0.9)
print(tok2.decode(out[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

What date is it today?" she asked.

"That is a puzzle," the lion said. "Do you want to keep it?"

"Yes!" Tim said.

The lion took the puzzle and set it into the backyard. Tim sat on the lion and said, "It is a very special puzzle. It is for the whole world and it shows us."

"Yay!" Tim said


In [9]:
import torch
import torch.nn.functional as F
from transformers import (
    GPT2Config, GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    TrainingArguments, Trainer
)
import json, os
from lion_pytorch import Lion

# ============ EXPERIMENT CONFIG ============
LEARNING_RATE   = 1e-3          # try: 1e-3, 5e-4, 3e-4, 1e-4
OPTIMIZER_NAME  = "adam"       # try: "adamw", "adam", "sgd"
LABEL_SMOOTHING = 0           # try: 0.0, 0.1, 0.2
RUN_NAME        = "finalised_best_model"
# ===========================================

data_collator = DataCollatorForLanguageModeling(tok, mlm=False)

config = GPT2Config(
    vocab_size=8000,
    n_positions=512,
    n_ctx=512,
    n_embd=768,
    n_layer=12,
    n_head=12,
    bos_token_id=tok.bos_token_id,
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.pad_token_id,
)

model = GPT2LMHeadModel(config)
print("Total parameters:", sum(p.numel() for p in model.parameters())/1e6, "M")


# ============ CUSTOM TRAINER ============
class CustomTrainer(Trainer):

    def create_optimizer(self):
        if OPTIMIZER_NAME == "adamw":
            self.optimizer = torch.optim.AdamW(
                self.model.parameters(),
                lr=self.args.learning_rate,
                weight_decay=self.args.weight_decay,
            )
        elif OPTIMIZER_NAME == "adam":
            self.optimizer = torch.optim.Adam(
                self.model.parameters(),
                lr=self.args.learning_rate,
            )
        elif OPTIMIZER_NAME == "sgd":
            self.optimizer = torch.optim.SGD(
                self.model.parameters(),
                lr=self.args.learning_rate,
                momentum=0.9,
                weight_decay=self.args.weight_decay,
            )
        elif OPTIMIZER_NAME == "lion":
            self.optimizer = Lion(
                self.model.parameters(),
                lr=self.args.learning_rate / 3,  # Lion needs ~3x smaller LR than Adam
                weight_decay=self.args.weight_decay,
            )
        return self.optimizer
# ========================================


training_args = TrainingArguments(
    output_dir=f"./runs/{RUN_NAME}",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,
    num_train_epochs=1,
    learning_rate=LEARNING_RATE,
    warmup_steps=500,              
    weight_decay=0.1,
    fp16=True,
    report_to="none",
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
    data_collator=data_collator,
)

trainer.train()

# Save final model + tokenizer
save_path = f"./runs/{RUN_NAME}/final_model"

trainer.save_model(save_path)   # saves model + trainer config
tok.save_pretrained(save_path)  # saves tokenizer files

print(f"Model saved to: {save_path}")

# ============ LOG RESULTS ============
eval_logs = [x for x in trainer.state.log_history if "eval_loss" in x]

if eval_logs:
    val_loss = eval_logs[-1]["eval_loss"]
    perplexity = torch.exp(torch.tensor(val_loss)).item()

    log_entry = {
        "run_name"        : RUN_NAME,
        "learning_rate"   : LEARNING_RATE,
        "optimizer"       : OPTIMIZER_NAME,
        "label_smoothing" : LABEL_SMOOTHING,
        "final_val_loss"  : val_loss,
        "perplexity"      : perplexity,
    }

    log_path = "./experiment_log.json"
    all_logs = json.load(open(log_path)) if os.path.exists(log_path) else []
    all_logs.append(log_entry)
    json.dump(all_logs, open(log_path, "w"), indent=2)

    print(f"\nRun:        {RUN_NAME}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Perplexity: {perplexity:.2f}")
else:
    print("No eval logs found — check eval_steps vs total training steps")
# =====================================

Total parameters: 91.593216 M


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 